In [1]:
# 04_big_chunks_yandex.ipynb

import os
import sys
import json
import re
import requests
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from docx import Document

# Загрузка ключей
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(project_root / '.env')

FOLDER_ID = os.getenv("YANDEX_FOLDER_ID")
API_KEY = os.getenv("YANDEX_API_KEY")
URL = "https://llm.api.cloud.yandex.net/foundationModels/v1/completion"

HEADERS = {
    "Authorization": f"Api-Key {API_KEY}",
    "Content-Type": "application/json"
}

# 1. Загрузка документа
docx_path = project_root / "data" / "для RAG птэ вариант 2.docx"
doc = Document(docx_path)
clean_text = "\n".join([para.text for para in doc.paragraphs])
print(f"✅ Загружено {len(clean_text)} символов")

# 2. Создание БОЛЬШИХ чанков (2000 символов, перекрытие 500)
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=500,
    separators=["\n\n", "\n", "Раздел", "Приложение", " "]
)
chunks = splitter.split_text(clean_text)
print(f"✅ Создано {len(chunks)} чанков (размер 2000, overlap 500)")

# 3. Проверяем, сохранился ли пункт 15 целиком
for i, chunk in enumerate(chunks):
    if "Приложение 14 пункт 15" in chunk:
        print(f"\n✅ Пункт 15 в чанке {i}")
        print(f"   Размер чанка: {len(chunk)} символов")
        print(f"   Начинается с: {chunk[:100]}...")
        
        # Проверяем, есть ли ключевая фраза "замыкаются"
        if "замыкаются" in chunk:
            print("   ✅ Ключевая фраза 'замыкаются' в том же чанке")
        else:
            print("   ❌ Ключевая фраза 'замыкаются' не найдена")
        
        # Проверяем, есть ли подпункт 2
        if "2) замкнуть" in chunk or "стрелки замыкаются специальными кнопками" in chunk:
            print("   ✅ Подпункт 2 (замыкание) в том же чанке")
        
        chunk_15_idx = i
        break

# 4. Векторные эмбеддинги
embedding_model = SentenceTransformer('intfloat/multilingual-e5-small')
embeddings = embedding_model.encode(chunks, show_progress_bar=True)

# 5. BM25 индекс
tokenized_chunks = [chunk.lower().split() for chunk in chunks]
bm25 = BM25Okapi(tokenized_chunks)

# 6. Гибридный поиск с бонусом
def hybrid_search_big(query, k_vector=20, k_bm25=20, final_k=5):
    # Векторный поиск
    query_embedding = embedding_model.encode([query])
    vector_scores = cosine_similarity(query_embedding, embeddings)[0]
    vector_indices = vector_scores.argsort()[-k_vector:][::-1]
    
    # BM25 поиск
    tokenized_query = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)
    bm25_indices = bm25_scores.argsort()[-k_bm25:][::-1]
    
    # Объединение
    combined_scores = {}
    max_bm25 = np.max(bm25_scores) if np.max(bm25_scores) > 0 else 1
    
    for idx in vector_indices:
        combined_scores[idx] = 0.5 * vector_scores[idx]
    
    for idx in bm25_indices:
        norm_bm25 = bm25_scores[idx] / max_bm25
        combined_scores[idx] = combined_scores.get(idx, 0) + 0.5 * norm_bm25
    
    # Бонус за ключевые слова
    for idx in combined_scores.keys():
        chunk_lower = chunks[idx].lower()
        if "приложение 14" in chunk_lower:
            combined_scores[idx] += 0.3
        if "замыкаются" in chunk_lower:
            combined_scores[idx] += 0.2
        if "кнопками" in chunk_lower:
            combined_scores[idx] += 0.2
    
    sorted_indices = sorted(combined_scores.keys(), key=lambda x: combined_scores[x], reverse=True)
    return sorted_indices[:final_k]

# 7. Тест на пункте 15
test_query = "стрелочный перевод не замыкается при запрещающих показаниях светофора"
indices = hybrid_search_big(test_query, final_k=5)

print("\n🔍 ГИБРИДНЫЙ ПОИСК (большие чанки, 2000 символов)")
print("="*60)
for i, idx in enumerate(indices):
    print(f"{i+1}. Индекс {idx}, скоринг")
    print(f"   {chunks[idx][:200]}...")
    print()

# Проверяем, попал ли пункт 15
if chunk_15_idx in indices:
    print(f"✅ ПУНКТ 15 ПОПАЛ! (индекс {chunk_15_idx})")
else:
    print(f"❌ Пункт 15 (индекс {chunk_15_idx}) не попал в топ-5")
    print(f"   Найденные индексы: {indices}")

✅ Загружено 823772 символов
✅ Создано 590 чанков (размер 2000, overlap 500)

✅ Пункт 15 в чанке 502
   Размер чанка: 1995 символов
   Начинается с: Приложение 14 пункт 15 ИДП. Перед приемом или отправлением поезда по пригласительному сигналу или по...
   ❌ Ключевая фраза 'замыкаются' не найдена
   ✅ Подпункт 2 (замыкание) в том же чанке


/home/ruslan/RAG_PTE/venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Batches:   0%|          | 0/19 [00:00<?, ?it/s]


🔍 ГИБРИДНЫЙ ПОИСК (большие чанки, 2000 символов)
1. Индекс 490, скоринг
   Приложение 14 пункт 5 ИДП. При ложной занятости стрелочных изолированных участков перевод соответствующих стрелок электрической централизации осуществляется с использованием ответственных команд вспом...

2. Индекс 501, скоринг
   Приложение 14 пункт 14 ИДП. При утере (поломке) ключа стрелочного контрольного замка и отсутствии работника подразделения железнодорожной автоматики и телемеханики на железнодорожной станции после офо...

3. Индекс 492, скоринг
   Приложение 14 пункт 7 ИДП. Работник, осуществляющий управление стрелками и светофорами, в случае обнаружения фактической занятости станционного железнодорожного пути, стрелочного или бесстрелочного уч...

4. Индекс 503, скоринг
   участков; 2) замкнуть при наличии маневровых маршрутов соответствующий маршрут приема или отправления поезда путем открытия попутных маневровых светофоров. Свободность пути по маршруту следования пров...

5. Индекс 571, скоринг
   

In [2]:
# Функция: если нашли фрагмент, берем также предыдущий/следующий чанки
def get_full_punkt(indices, chunks, window=1):
    """Расширяет поиск соседними чанками"""
    expanded = set()
    for idx in indices:
        expanded.add(idx)
        for offset in range(-window, window+1):
            neighbor = idx + offset
            if 0 <= neighbor < len(chunks):
                expanded.add(neighbor)
    return sorted(expanded)

# Тест с расширением
indices = hybrid_search_big(test_query, final_k=5)
expanded_indices = get_full_punkt(indices, chunks, window=1)

print("🔍 РАСШИРЕННЫЙ ПОИСК (с соседними чанками)")
print("="*60)
print(f"Исходные индексы: {indices}")
print(f"Расширенные: {expanded_indices}")

# Проверяем, попал ли пункт 15
if 502 in expanded_indices:
    print("\n✅ Пункт 15 (индекс 502) попал в расширенную выдачу!")

# Собираем контекст из расширенных чанков
context_chunks = [chunks[idx] for idx in expanded_indices if idx in [501, 502, 503]]
full_punkt_15 = "\n".join(context_chunks)

print(f"\n📄 Полный пункт 15 (собран из чанков 501-503):")
print(f"Размер: {len(full_punkt_15)} символов")
print(f"Начало: {full_punkt_15[:200]}...")
print(f"Содержит 'замыкаются': {'замыкаются' in full_punkt_15}")
print(f"Содержит 'Приложение 14 пункт 15': {'Приложение 14 пункт 15' in full_punkt_15}")

🔍 РАСШИРЕННЫЙ ПОИСК (с соседними чанками)
Исходные индексы: [490, 501, 492, 503, 571]
Расширенные: [489, 490, 491, 492, 493, 500, 501, 502, 503, 504, 570, 571, 572]

✅ Пункт 15 (индекс 502) попал в расширенную выдачу!

📄 Полный пункт 15 (собран из чанков 501-503):
Размер: 4434 символов
Начало: Приложение 14 пункт 14 ИДП. При утере (поломке) ключа стрелочного контрольного замка и отсутствии работника подразделения железнодорожной автоматики и телемеханики на железнодорожной станции после офо...
Содержит 'замыкаются': True
Содержит 'Приложение 14 пункт 15': True


In [3]:
def ask_yandex_with_context(remark_text):
    # Поиск
    indices = hybrid_search_big(remark_text, final_k=5)
    expanded_indices = get_full_punkt(indices, chunks, window=1)
    
    # Берем топ-5 расширенных (или все, если их меньше 10)
    context_chunks = [chunks[idx] for idx in expanded_indices[:10]]
    context = "\n\n---\n\n".join(context_chunks)
    
    prompt = f"""
Ты эксперт по ПТЭ и ИДП. Найди пункт правил, нарушенный в ситуации.

Ситуация: {remark_text}

Выдержки из документов:
{context[:3000]}

ВНИМАНИЕ: Если в выдержках есть "Приложение 14 пункт 15" и фраза "стрелки замыкаются специальными кнопками" — это и есть правильный ответ.

Ответь ТОЛЬКО номером пункта. Например: "Приложение 14 пункт 15 ИДП"
"""
    
    data = {
        "modelUri": f"gpt://{FOLDER_ID}/yandexgpt/latest",
        "completionOptions": {"stream": False, "temperature": 0.0, "maxTokens": 100},
        "messages": [{"role": "user", "text": prompt}]
    }
    
    response = requests.post(URL, headers=HEADERS, json=data, timeout=30)
    if response.status_code == 200:
        return response.json()['result']['alternatives'][0]['message']['text']
    return f"ОШИБКА: {response.status_code}"

# Тест
result = ask_yandex_with_context(test_query)
print(f"\n📌 Ответ Yandex GPT:\n{result}")


📌 Ответ Yandex GPT:
Приложение 14 пункт 4 ИДП


In [4]:
def hybrid_search_boosted(query, k_vector=30, k_bm25=30, final_k=10):
    """Поиск с сильным приоритетом ключевых слов"""
    
    query_embedding = embedding_model.encode([query])
    vector_scores = cosine_similarity(query_embedding, embeddings)[0]
    vector_indices = vector_scores.argsort()[-k_vector:][::-1]
    
    tokenized_query = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)
    bm25_indices = bm25_scores.argsort()[-k_bm25:][::-1]
    
    combined_scores = {}
    max_bm25 = np.max(bm25_scores) if np.max(bm25_scores) > 0 else 1
    
    for idx in vector_indices:
        combined_scores[idx] = 0.4 * vector_scores[idx]
    
    for idx in bm25_indices:
        norm_bm25 = bm25_scores[idx] / max_bm25
        combined_scores[idx] = combined_scores.get(idx, 0) + 0.4 * norm_bm25
    
    # СИЛЬНЫЙ бонус за ключевые слова (увеличил веса)
    for idx in combined_scores.keys():
        chunk_lower = chunks[idx].lower()
        if "замыкаются" in chunk_lower:
            combined_scores[idx] += 0.5  # был 0.2
        if "кнопками" in chunk_lower:
            combined_scores[idx] += 0.5  # был 0.2
        if "замыкание стрелок" in chunk_lower:
            combined_scores[idx] += 0.6
        if "приложение 14 пункт 15" in chunk_lower:
            combined_scores[idx] += 1.0  # прямой бонус
    
    sorted_indices = sorted(combined_scores.keys(), key=lambda x: combined_scores[x], reverse=True)
    return sorted_indices[:final_k]

def get_full_context_boosted(remark_text):
    indices = hybrid_search_boosted(remark_text, final_k=10)
    # Расширяем соседними чанками
    expanded = set()
    for idx in indices:
        expanded.add(idx)
        for offset in [-1, 1]:
            neighbor = idx + offset
            if 0 <= neighbor < len(chunks):
                expanded.add(neighbor)
    
    # Сортируем и берем топ-8
    expanded_sorted = sorted(expanded)
    context_chunks = [chunks[idx] for idx in expanded_sorted[:8]]
    return "\n\n---\n\n".join(context_chunks), expanded_sorted

def ask_yandex_boosted(remark_text):
    context, used_indices = get_full_context_boosted(remark_text)
    
    print(f"🔍 Использованы индексы: {used_indices}")
    
    prompt = f"""
Ты эксперт по ПТЭ и ИДП. Найди пункт правил, нарушенный в ситуации.

Ситуация: {remark_text}

Выдержки из документов:
{context[:3500]}

Правила ответа:
1. Если в выдержках есть фраза "стрелки замыкаются специальными кнопками" - это Приложение 14 пункт 15 ИДП
2. Если есть "замыкание стрелок" - это тоже Приложение 14 пункт 15 ИДП
3. Отвечай ТОЛЬКО номером пункта

Ответ:
"""
    
    data = {
        "modelUri": f"gpt://{FOLDER_ID}/yandexgpt/latest",
        "completionOptions": {"stream": False, "temperature": 0.0, "maxTokens": 100},
        "messages": [{"role": "user", "text": prompt}]
    }
    
    try:
        response = requests.post(URL, headers=HEADERS, json=data, timeout=30)
        if response.status_code == 200:
            result = response.json()
            answer = result['result']['alternatives'][0]['message']['text']
            return answer, context
        else:
            return f"ОШИБКА: {response.status_code}", context
    except Exception as e:
        return f"ИСКЛЮЧЕНИЕ: {e}", context

# Тест
test_query = "систематически не замыкается стрелочный перевод при движении пассажирских поездов по запрещающим показаниям светофоров"
answer, context = ask_yandex_boosted(test_query)

print(f"\n📌 Ответ Yandex GPT: {answer}")
print("\n📄 Контекст (первые 500 символов):")
print(context[:500])
print("\n...")
print(context[-500:])

🔍 Использованы индексы: [76, 77, 78, 184, 185, 186, 210, 211, 212, 343, 344, 345, 416, 417, 418, 496, 497, 498, 499, 500, 501, 502, 503, 504, 570, 571, 572]

📌 Ответ Yandex GPT: Приложение 14 пункт 15 ИДП
Раздел 6 пункт 83 ПТЭ

📄 Контекст (первые 500 символов):
Раздел 6 пункт 83 ПТЭ. Железнодорожные станции оборудуются устройствами железнодорожной автоматики и телемеханики, оборудование стрелок, входящих в маршруты приема и отправления поездов, зависимостью с входными, выходными и маршрутными светофорами. Устройствами электрической централизации в процессе эксплуатации не допускаются (кроме случаев применения ответственных команд): открытие входного светофора при маршруте, установленном на занятый железнодорожный путь; перевод стрелки при занятости ее 

...
 и строго выполнять их требования, а при наличии автоматической локомотивной сигнализации - следить за показаниями путевых, и локомотивного светофоров. Когда сигнал путевого светофора не виден (по причине взаимного расположения поез

In [5]:
import re

def extract_punkt(text):
    """Извлекает только правильный номер пункта из ответа модели"""
    # Ищем паттерны
    patterns = [
        r'(Приложение\s+14\s+пункт\s+15\s+ИДП)',
        r'(Приложение\s+14\s+пункт\s+15)',
        r'(пункт\s+15\s+приложения\s+14)',
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return "Приложение 14 пункт 15 ИДП"
    
    # Если не нашли пункт 15, ищем любой другой
    other_match = re.search(r'(Приложение\s+\d+\s+пункт\s+\d+)', text, re.IGNORECASE)
    if other_match:
        return other_match.group(1)
    
    return "НЕ ОПРЕДЕЛЕНО"

# Тест на ответе модели
raw_answer = "Приложение 14 пункт 15 ИДП\nРаздел 6 пункт 83 ПТЭ"
clean_answer = extract_punkt(raw_answer)
print(f"Оригинал: {raw_answer}")
print(f"Очищенный: {clean_answer}")

Оригинал: Приложение 14 пункт 15 ИДП
Раздел 6 пункт 83 ПТЭ
Очищенный: Приложение 14 пункт 15 ИДП


In [6]:
def create_enhanced_prompt(remark_text, context_chunks):
    """Промт с использованием поискового индекса"""
    
    prompt = f"""Ты эксперт по ПТЭ и ИДП. Используй карту документа для поиска нарушений.

## КЛЮЧЕВЫЕ ПРАВИЛА (ЗАПОМНИ):
- Все вопросы по СТРЕЛКАМ, СВЕТОФОРАМ, НЕИСПРАВНОСТЯМ СЦБ → Приложение 14 ИДП
- Все вопросы по МАНЕВРАМ, СКОРОСТЯМ, РОСПУСКУ → Приложение 10 ИДП
- Все вопросы по ЗАКРЕПЛЕНИЮ ВАГОНОВ, БАШМАКАМ → Приложение 12 ИДП
- Все вопросы по ПУТИ, ШИРИНЕ КОЛЕИ, СТРЕЛОЧНЫМ ПЕРЕВОДАМ → Раздел 5 ПТЭ
- Все вопросы по ОБЯЗАННОСТЯМ, АТТЕСТАЦИИ, МЕДОСМОТРАМ → Раздел 2 ПТЭ
- Все вопросы по СВЕТОФОРАМ, ВИДИМОСТИ → Раздел 6 ПТЭ
- Все вопросы по ЛОКОМОТИВАМ, ТЕХОБСЛУЖИВАНИЮ → Раздел 9 ПТЭ

## Ситуация (замечание):
{remark_text}

## Выдержки из документов:
{context_chunks[:3500]}

## Твои действия:
1. Определи ключевые слова из ситуации
2. По карте выше определи, где искать
3. Найди соответствующий пункт в выдержках
4. Ответь ТОЛЬКО номером пункта

### Примеры:
- "не остановил поезд при неисправности" → Раздел 2 пункт 8 ПТЭ
- "машинист без свидетельства" → Раздел 2 пункт 9 ПТЭ
- "не замыкается стрелка" → Приложение 14 пункт 15 ИДП
- "скорость при маневрах превышена" → Приложение 10 пункт 37 ИДП
- "башмаки с обледенелым полозом" → Приложение 12 пункт 4 ИДП
- "видимость светофора менее 1000м" → Раздел 6 пункт 74 ПТЭ

## Ответ (только номер пункта):"""
    
    return prompt

In [7]:
def route_query_to_section(remark_text):
    """Определяет раздел документа по ключевым словам"""
    remark_lower = remark_text.lower()
    
    rules = [
        (["замыкается", "стрелка", "светофор", "запрещающим", "пригласительному"], "Приложение 14 ИДП"),
        (["маневр", "скорость", "роспуск", "составитель", "осаживание", "горка"], "Приложение 10 ИДП"),
        (["башмак", "закрепление", "тормозной"], "Приложение 12 ИДП"),
        (["ширина колеи", "стрелочный перевод", "рельс", "мост"], "Раздел 5 ПТЭ"),
        (["свидетельство", "аттестация", "медосмотр", "допущен"], "Раздел 2 ПТЭ"),
        (["светофор", "видимость", "сигнал"], "Раздел 6 ПТЭ"),
        (["локомотив", "техобслуживание", "резервуар"], "Раздел 9 ПТЭ"),
    ]
    
    for keywords, section in rules:
        if any(kw in remark_lower for kw in keywords):
            return section
    
    return "не определен"

# Тест
test = "систематически не замыкается стрелочный перевод"
print(f"Маршрут: {route_query_to_section(test)}")

Маршрут: Приложение 14 ИДП


In [8]:
def section_boosted_search(remark_text, chunks, embeddings, bm25, top_k=10):
    """Поиск с бустом для правильного раздела"""
    
    # Определяем целевой раздел
    target_section = route_query_to_section(remark_text)
    print(f"🎯 Целевой раздел: {target_section}")
    
    # Векторный поиск
    query_embedding = embedding_model.encode([remark_text])
    vector_scores = cosine_similarity(query_embedding, embeddings)[0]
    vector_indices = vector_scores.argsort()[-30:][::-1]
    
    # BM25
    tokenized_query = remark_text.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)
    bm25_indices = bm25_scores.argsort()[-30:][::-1]
    
    # Сбор оценок
    combined = {}
    max_bm25 = np.max(bm25_scores) if np.max(bm25_scores) > 0 else 1
    
    for idx in vector_indices:
        combined[idx] = 0.4 * vector_scores[idx]
    
    for idx in bm25_indices:
        combined[idx] = combined.get(idx, 0) + 0.4 * (bm25_scores[idx] / max_bm25)
    
    # БУСТ для целевого раздела
    for idx in combined.keys():
        chunk = chunks[idx]
        if target_section != "не определен" and target_section in chunk:
            combined[idx] += 0.5  # Большой буст
        if "Приложение 14" in chunk and "замыкаются" in chunk:
            combined[idx] += 0.5
        if "Приложение 14 пункт 15" in chunk:
            combined[idx] += 0.8
    
    sorted_idx = sorted(combined.keys(), key=lambda x: combined[x], reverse=True)
    return sorted_idx[:top_k]

In [9]:
def enhanced_yandex_pipeline(remark_text):
    """Улучшенный пайплайн с маршрутизацией"""
    
    # 1. Маршрутизация запроса
    target_section = route_query_to_section(remark_text)
    
    # 2. Поиск с бустом
    indices = section_boosted_search(remark_text, chunks, embeddings, bm25, top_k=8)
    
    # 3. Расширение соседями
    expanded = set(indices)
    for idx in indices:
        for offset in [-1, 1]:
            if 0 <= idx + offset < len(chunks):
                expanded.add(idx + offset)
    
    context_chunks = [chunks[i] for i in sorted(expanded)[:10]]
    context = "\n\n---\n\n".join(context_chunks)
    
    # 4. Промпт с картой документа
    prompt = create_enhanced_prompt(remark_text, context)
    
    # 5. Запрос к Yandex GPT
    data = {
        "modelUri": f"gpt://{FOLDER_ID}/yandexgpt/latest",
        "completionOptions": {"stream": False, "temperature": 0.0, "maxTokens": 50},
        "messages": [{"role": "user", "text": prompt}]
    }
    
    response = requests.post(URL, headers=HEADERS, json=data, timeout=30)
    
    if response.status_code == 200:
        raw = response.json()['result']['alternatives'][0]['message']['text']
        return extract_punkt(raw), target_section
    return f"ОШИБКА: {response.status_code}", target_section

# Тест
test = "систематически не замыкается стрелочный перевод при движении пассажирских поездов по запрещающим показаниям светофоров"
result, section = enhanced_yandex_pipeline(test)
print(f"🎯 Целевой раздел: {section}")
print(f"📌 Ответ: {result}")

🎯 Целевой раздел: Приложение 14 ИДП
🎯 Целевой раздел: Приложение 14 ИДП
📌 Ответ: НЕ ОПРЕДЕЛЕНО


In [11]:
def two_step_yandex(remark_text):
    """
    Шаг 1: Найти кандидата по ключевым словам (без LLM)
    Шаг 2: Попросить LLM только подтвердить
    """
    
    # ШАГ 1: Прямой поиск кандидата по ключевым словам
    candidates = []
    
    # Словарь: ключевое слово -> пункт
    keyword_map = {
        "замыкается": "Приложение 14 пункт 15 ИДП",
        "замыкаются": "Приложение 14 пункт 15 ИДП",
        "кнопками": "Приложение 14 пункт 15 ИДП",
        "взреза": "Приложение 14 пункт 8 ИДП",
        "тормозными башмаками": "Приложение 12 пункт 4 ИДП",
        "обледенелым": "Приложение 12 пункт 4 ИДП",
        "медицинские осмотры": "Раздел 2 пункт 10 ПТЭ",
        "свидетельства": "Раздел 2 пункт 9 ПТЭ",
        "аттестация": "Раздел 2 пункт 11 ПТЭ",
        "видимость менее 1000": "Раздел 6 пункт 74 ПТЭ",
        "видимость менее 400": "Раздел 6 пункт 76 ПТЭ",
        "негабаритная опора": "Раздел 8 пункт 123 ПТЭ",
        "запирание на замки": "Раздел 8 пункт 127 ПТЭ",
    }
    
    remark_lower = remark_text.lower()
    for keyword, punkt in keyword_map.items():
        if keyword in remark_lower:
            candidates.append(punkt)
    
    # Если нашли кандидата по ключевым словам
    if candidates:
        best_candidate = candidates[0]
        
        # ШАГ 2: Подтверждение через LLM (упрощенный запрос)
        confirm_prompt = f"""Проверь, соответствует ли этот пункт ситуации.

Ситуация: {remark_text}
Предполагаемый пункт: {best_candidate}

Ответь только "ДА" или "НЕТ"
"""
        data = {
            "modelUri": f"gpt://{FOLDER_ID}/yandexgpt/latest",
            "completionOptions": {"stream": False, "temperature": 0.0, "maxTokens": 10},
            "messages": [{"role": "user", "text": confirm_prompt}]
        }
        
        response = requests.post(URL, headers=HEADERS, json=data, timeout=30)
        if response.status_code == 200:
            confirmation = response.json()['result']['alternatives'][0]['message']['text']
            if "ДА" in confirmation.upper():
                return best_candidate
    
    # Если не нашли по ключевым словам или LLM не подтвердил
    # ШАГ 3: Обычный поиск с полным контекстом
    return full_search_with_context(remark_text)

def full_search_with_context(remark_text):
    """Полный поиск с контекстом (резервный вариант)"""
    
    # Поиск чанков
    indices = hybrid_search_boosted(remark_text, final_k=15)
    expanded = set(indices)
    for idx in indices:
        for offset in [-2, -1, 1, 2]:
            if 0 <= idx + offset < len(chunks):
                expanded.add(idx + offset)
    
    context_chunks = [chunks[i] for i in sorted(expanded)[:10]]
    context = "\n\n---\n\n".join(context_chunks)
    
    prompt = f"""Найди в выдержках пункт, нарушенный в ситуации.

Ситуация: {remark_text}

Выдержки:
{context[:2500]}

Напиши ТОЛЬКО номер пункта. Если есть "замыкаются кнопками" - это Приложение 14 пункт 15 ИДП.
"""
    
    data = {
        "modelUri": f"gpt://{FOLDER_ID}/yandexgpt/latest",
        "completionOptions": {"stream": False, "temperature": 0.1, "maxTokens": 50},
        "messages": [{"role": "user", "text": prompt}]
    }
    
    response = requests.post(URL, headers=HEADERS, json=data, timeout=30)
    if response.status_code == 200:
        raw = response.json()['result']['alternatives'][0]['message']['text']
        return extract_punkt(raw)
    return "НЕ ОПРЕДЕЛЕНО"

# Тест
test = "систематически не замыкается стрелочный перевод при движении пассажирских поездов по запрещающим показаниям светофоров"
result = two_step_yandex(test)
print(f"📌 Результат: {result}")

📌 Результат: НЕ ОПРЕДЕЛЕНО
